In [ ]:
import os
import json
import math
import time
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings("ignore")

SEED = 7

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("TORCH:", torch.__version__)
print("CUDA AVAILABLE:", torch.cuda.is_available())

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input/datasets/katakuricharlotte")
WORK_ROOT = Path("/kaggle/working/notebook12_outputs")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

PLANTVILLAGE_ROOT = INPUT_ROOT / "plantvillage" / "PlantVillage"
CROP_ROOT = INPUT_ROOT / "crop-manifestfiles"

PATHS = {
    "selected_manifest": CROP_ROOT / "plantvillage_selected_manifest.csv",
    "selected_manifest_clean": CROP_ROOT / "plantvillage_selected_manifest_clean.csv",
    "eda_clean_manifest": CROP_ROOT / "Eda-tables" / "Eda-tables" / "plantvillage_selected_manifest_clean.csv",
    "near_dup_candidates": CROP_ROOT / "Eda-tables" / "Eda-tables" / "near_duplicate_candidates_phash_hamming_le4.csv",
    "excluded_exact_dups": CROP_ROOT / "Eda-tables" / "Eda-tables" / "excluded_exact_train_val_duplicates.csv",
    "locked_master_manifest": CROP_ROOT / "split_outputs" / "split_outputs" / "manifests" / "plantvillage_master_manifest_locked.csv",
    "main19_train": CROP_ROOT / "split_outputs" / "split_outputs" / "manifests" / "main19" / "main19_train.csv",
    "main19_calibration": CROP_ROOT / "split_outputs" / "split_outputs" / "manifests" / "main19" / "main19_calibration.csv",
    "main19_internal_test": CROP_ROOT / "split_outputs" / "split_outputs" / "manifests" / "main19" / "main19_internal_test.csv",
    "class_to_index_main19": CROP_ROOT / "split_outputs" / "split_outputs" / "metadata" / "class_to_index_main19.json",
    "index_to_class_main19": CROP_ROOT / "split_outputs" / "split_outputs" / "metadata" / "index_to_class_main19.json",
    "excluded_confirmed_near_dup": CROP_ROOT / "split_outputs" / "split_outputs" / "tables" / "excluded_confirmed_near_duplicate_leakage.csv",
    "review_progress": CROP_ROOT / "split_outputs" / "split_outputs" / "tables" / "near_duplicate_manual_review_progress.csv",
    "review_decisions": CROP_ROOT / "split_outputs" / "split_outputs" / "tables" / "near_duplicate_review_decisions.csv",
    "locked_split_summary": CROP_ROOT / "split_outputs" / "split_outputs" / "tables" / "split_summary_main19_locked.csv",
}

missing = [k for k, v in PATHS.items() if not v.exists()]
print("Missing paths:", missing)
for k, v in PATHS.items():
    print(f"{k:30s} -> {v}")

In [ ]:
def inspect_csv(path, n=3):
    df = pd.read_csv(path, nrows=n)
    print(f"\nFILE: {path}")
    print("SHAPE SAMPLE:", df.shape)
    print("COLUMNS:", list(df.columns))
    display(df.head(n))
    return df

for key in [
    "main19_train",
    "main19_calibration",
    "main19_internal_test",
    "near_dup_candidates",
    "review_progress",
    "review_decisions",
    "excluded_confirmed_near_dup",
    "locked_master_manifest",
]:
    inspect_csv(PATHS[key], n=3)

In [ ]:
train_df = pd.read_csv(PATHS["main19_train"])
calib_df = pd.read_csv(PATHS["main19_calibration"])
test_df = pd.read_csv(PATHS["main19_internal_test"])

near_dup_df = pd.read_csv(PATHS["near_dup_candidates"])
review_progress_df = pd.read_csv(PATHS["review_progress"])
review_decisions_df = pd.read_csv(PATHS["review_decisions"])
excluded_near_dup_df = pd.read_csv(PATHS["excluded_confirmed_near_dup"])
locked_master_df = pd.read_csv(PATHS["locked_master_manifest"])
selected_clean_df = pd.read_csv(PATHS["selected_manifest_clean"])

with open(PATHS["class_to_index_main19"], "r", encoding="utf-8") as f:
    class_to_index_main19 = json.load(f)

with open(PATHS["index_to_class_main19"], "r", encoding="utf-8") as f:
    index_to_class_main19 = json.load(f)

print("train:", train_df.shape)
print("calibration:", calib_df.shape)
print("internal_test:", test_df.shape)
print("near_dup candidates:", near_dup_df.shape)
print("review_progress:", review_progress_df.shape)
print("review_decisions:", review_decisions_df.shape)
print("excluded_confirmed_near_dup:", excluded_near_dup_df.shape)
print("locked_master:", locked_master_df.shape)
print("selected_clean:", selected_clean_df.shape)
print("classes:", len(class_to_index_main19))

In [ ]:
def lower_map_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out

def first_existing(df, candidates, required=False, df_name="df"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Missing required column in {df_name}. Tried: {candidates}")
    return None

for name, obj in [
    ("train_df", train_df),
    ("calib_df", calib_df),
    ("test_df", test_df),
    ("near_dup_df", near_dup_df),
    ("review_progress_df", review_progress_df),
    ("review_decisions_df", review_decisions_df),
    ("excluded_near_dup_df", excluded_near_dup_df),
    ("locked_master_df", locked_master_df),
    ("selected_clean_df", selected_clean_df),
]:
    locals()[name] = lower_map_columns(obj)

manifest_example = train_df.copy()

COLS = {
    "manifest_image_id": first_existing(manifest_example, ["image_id", "id", "sample_id", "uuid"], required=False, df_name="manifest"),
    "manifest_relpath": first_existing(manifest_example, ["relative_path", "rel_path", "filepath", "file_path", "path"], required=False, df_name="manifest"),
    "manifest_split": first_existing(manifest_example, ["split", "partition"], required=False, df_name="manifest"),
    "manifest_class": first_existing(manifest_example, ["class_name", "label_name", "class", "label"], required=False, df_name="manifest"),
    "manifest_label_idx": first_existing(manifest_example, ["label_index", "class_index", "target", "label_id"], required=False, df_name="manifest"),
}

COLS

In [ ]:
import pandas as pd


def harmonize_manifest(df, source_name):
    df = df.copy().reset_index(drop=True)
    df.columns = [str(c).strip() for c in df.columns]

    image_id_col = first_existing(
        df,
        ["image_id", "id", "sample_id", "uuid"],
        required=False,
        df_name=source_name,
    )

    relpath_col = first_existing(
        df,
        ["relative_path", "rel_path", "filepath", "file_path", "path"],
        required=True,
        df_name=source_name,
    )

    class_col = first_existing(
        df,
        ["class_name", "label_name", "class", "label"],
        required=True,
        df_name=source_name,
    )

    label_idx_col = first_existing(
        df,
        ["label_index", "class_index", "target", "label_id"],
        required=False,
        df_name=source_name,
    )

    # Validate before converting missing values into strings.
    for col in [relpath_col, class_col]:
        invalid = df[col].isna() | df[col].astype(str).str.strip().eq("")
        if invalid.any():
            raise ValueError(
                f"{source_name}: {invalid.sum()} missing/empty values in {col!r}"
            )

    out = pd.DataFrame({
        "relative_path": (
            df[relpath_col]
            .astype(str)
            .str.strip()
            .str.replace("\\", "/", regex=False)
        ),
        "class_name": df[class_col].astype(str).str.strip(),
    })

    # Identify the file record, not its image content.
    if image_id_col is not None:
        invalid_ids = (
            df[image_id_col].isna()
            | df[image_id_col].astype(str).str.strip().eq("")
        )
        if invalid_ids.any():
            raise ValueError(f"{source_name}: missing/empty image IDs")

        out["image_id"] = df[image_id_col].astype(str).str.strip()
    else:
        out["image_id"] = out["relative_path"]
        print(f"{source_name}: using relative_path as image_id")

    mapped_labels = out["class_name"].map(class_to_index_main19)

    if mapped_labels.isna().any():
        unknown = sorted(
            out.loc[mapped_labels.isna(), "class_name"].unique().tolist()
        )
        raise ValueError(
            f"{source_name}: classes missing from class_to_index_main19: "
            f"{unknown}"
        )

    out["label_index"] = mapped_labels.astype(int)

    # Verify existing indices against the frozen class map, if present.
    if label_idx_col is not None:
        existing = pd.to_numeric(df[label_idx_col], errors="raise")
        mismatch = existing.isna() | existing.ne(out["label_index"])

        if mismatch.any():
            raise ValueError(
                f"{source_name}: {mismatch.sum()} label indices disagree "
                "with class_to_index_main19"
            )

    out["historical_split"] = source_name

    # Retain useful audit metadata without using hashes as file IDs.
    for col in ["image_path", "sha256", "phash", "file_name"]:
        if col in df.columns:
            out[col] = df[col]

    return out


train_m = harmonize_manifest(train_df, "train")
calib_m = harmonize_manifest(calib_df, "calibration")
test_m = harmonize_manifest(test_df, "internal_test")

locked_main19 = pd.concat(
    [train_m, calib_m, test_m],
    ignore_index=True,
)

# Do not silently discard repeated file records.
duplicate_paths = locked_main19["relative_path"].duplicated(keep=False)

if duplicate_paths.any():
    display(
        locked_main19.loc[
            duplicate_paths,
            ["image_id", "relative_path", "historical_split"],
        ].sort_values("relative_path")
    )
    raise ValueError(
        "Repeated relative paths found in the combined manifests. "
        "Review them before proceeding."
    )

print("Combined manifest shape:", locked_main19.shape)
print("Unique classes:", locked_main19["class_name"].nunique())

display(locked_main19.head())
display(locked_main19["historical_split"].value_counts())
display(locked_main19["class_name"].value_counts().head())

In [ ]:
unique_classes = sorted(locked_main19["class_name"].unique().tolist())
mapped_classes = sorted(class_to_index_main19.keys())

print("Manifest unique classes:", len(unique_classes))
print("Class-map keys:", len(mapped_classes))

missing_in_map = sorted(set(unique_classes) - set(mapped_classes))
missing_in_manifest = sorted(set(mapped_classes) - set(unique_classes))

print("Missing in class map:", missing_in_map)
print("Missing in manifest:", missing_in_manifest)

if missing_in_map:
    raise ValueError("Some manifest classes are missing from class_to_index_main19.json")

locked_main19["label_index_from_map"] = locked_main19["class_name"].map(class_to_index_main19)

mismatch = locked_main19.loc[locked_main19["label_index"] != locked_main19["label_index_from_map"]]
print("Label index mismatches:", len(mismatch))
display(mismatch.head())

In [ ]:
def resolve_image_path(rel_path):
    rel_path = str(rel_path).strip().replace("\\", "/")
    candidates = [
        PLANTVILLAGE_ROOT / rel_path,
        PLANTVILLAGE_ROOT / "train" / rel_path,
        PLANTVILLAGE_ROOT / "val" / rel_path,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

locked_main19["resolved_path"] = locked_main19["relative_path"].apply(resolve_image_path)
missing_paths = locked_main19["resolved_path"].isna().sum()

print("Missing resolved paths:", missing_paths)
display(locked_main19[locked_main19["resolved_path"].isna()].head(20))

In [ ]:
locked_main19["image_id"] = locked_main19["image_id"].astype(str)
locked_main19 = locked_main19.drop_duplicates(subset=["image_id"]).copy()

print("Unique images in locked Main19:", len(locked_main19))
display(locked_main19["historical_split"].value_counts())

In [ ]:
near_dup_df.columns.tolist()

In [ ]:
def normalize_path(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace("\\", "/", regex=False)
    )


def harmonize_candidate_pairs(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    train_path_col = first_existing(
        df,
        ["train_image_path", "image_a_path", "path_a", "image_path_a"],
        required=True,
        df_name="near_duplicate_candidates",
    )

    val_path_col = first_existing(
        df,
        ["val_image_path", "image_b_path", "path_b", "image_path_b"],
        required=True,
        df_name="near_duplicate_candidates",
    )

    distance_col = first_existing(
        df,
        ["phash_hamming_distance", "hamming_distance", "distance"],
        required=True,
        df_name="near_duplicate_candidates",
    )

    label_col = first_existing(
        df,
        ["label", "class_name", "original_label"],
        required=False,
        df_name="near_duplicate_candidates",
    )

    train_file_col = first_existing(
        df,
        ["train_file_name", "file_name_a"],
        required=False,
        df_name="near_duplicate_candidates",
    )

    val_file_col = first_existing(
        df,
        ["val_file_name", "file_name_b"],
        required=False,
        df_name="near_duplicate_candidates",
    )

    train_phash_col = first_existing(
        df,
        ["train_phash", "phash_a"],
        required=False,
        df_name="near_duplicate_candidates",
    )

    val_phash_col = first_existing(
        df,
        ["val_phash", "phash_b"],
        required=False,
        df_name="near_duplicate_candidates",
    )

    out = pd.DataFrame({
        # These are full Kaggle paths from the original candidate CSV.
        "image_path_a": normalize_path(df[train_path_col]),
        "image_path_b": normalize_path(df[val_path_col]),
        "phash_hamming_distance": pd.to_numeric(
            df[distance_col], errors="raise"
        ).astype(int),
    })

    # Use paths as stable pair identifiers because this CSV has no image_id fields.
    out["image_id_a"] = out["image_path_a"]
    out["image_id_b"] = out["image_path_b"]

    if label_col is not None:
        out["label"] = df[label_col].astype(str).str.strip()

    if train_file_col is not None:
        out["file_name_a"] = df[train_file_col].astype(str)

    if val_file_col is not None:
        out["file_name_b"] = df[val_file_col].astype(str)

    if train_phash_col is not None:
        out["phash_a"] = df[train_phash_col].astype(str)

    if val_phash_col is not None:
        out["phash_b"] = df[val_phash_col].astype(str)

    return out


near_pairs = harmonize_candidate_pairs(near_dup_df)

print("near_pairs shape:", near_pairs.shape)
print("Columns:", near_pairs.columns.tolist())
display(near_pairs.head())

In [ ]:
path_to_id = dict(zip(locked_main19["relative_path"].astype(str), locked_main19["image_id"].astype(str)))
id_to_path = dict(zip(locked_main19["image_id"].astype(str), locked_main19["relative_path"].astype(str)))

if "image_id_a" not in near_pairs.columns and "relative_path_a" in near_pairs.columns:
    near_pairs["image_id_a"] = near_pairs["relative_path_a"].map(path_to_id)

if "image_id_b" not in near_pairs.columns and "relative_path_b" in near_pairs.columns:
    near_pairs["image_id_b"] = near_pairs["relative_path_b"].map(path_to_id)

if "relative_path_a" not in near_pairs.columns and "image_id_a" in near_pairs.columns:
    near_pairs["relative_path_a"] = near_pairs["image_id_a"].map(id_to_path)

if "relative_path_b" not in near_pairs.columns and "image_id_b" in near_pairs.columns:
    near_pairs["relative_path_b"] = near_pairs["image_id_b"].map(id_to_path)

print("Missing image_id_a:", near_pairs["image_id_a"].isna().sum() if "image_id_a" in near_pairs.columns else "column missing")
print("Missing image_id_b:", near_pairs["image_id_b"].isna().sum() if "image_id_b" in near_pairs.columns else "column missing")
display(near_pairs.head())

In [ ]:
# CELL: Build near_pairs, then filter Main19-relevant candidate pairs by path match

near_pairs = review_progress_df.copy()

near_pairs = near_pairs.rename(
    columns={
        "candidate_id": "pair_key",
        "original_label": "label",
        "review_decision": "decision",
        "review_reason": "reason",
        "image_a_path": "image_path_a",
        "image_b_path": "image_path_b",
    }
)

def to_dataset_relative_path(series):
    """
    Convert either:
      /kaggle/input/.../plantvillage/train/Class/file.jpg
    or:
      train/Class/file.jpg
    into the comparable form:
      train/Class/file.jpg
    """
    s = (
        series.astype(str)
        .str.strip()
        .str.replace("\\", "/", regex=False)
    )

    extracted = s.str.extract(r"(?i)(?:^|/)((?:train|val)/.*)$", expand=False)
    return extracted.fillna(s).str.lower()

locked_main19["relative_path_key"] = to_dataset_relative_path(
    locked_main19["relative_path"]
)

near_pairs["relative_path_a"] = to_dataset_relative_path(
    near_pairs["image_path_a"]
)

near_pairs["relative_path_b"] = to_dataset_relative_path(
    near_pairs["image_path_b"]
)

main19_relative_paths = set(locked_main19["relative_path_key"])

near_pairs_main19 = near_pairs[
    near_pairs["relative_path_a"].isin(main19_relative_paths)
    | near_pairs["relative_path_b"].isin(main19_relative_paths)
].copy()

print("Total candidate pairs:", len(near_pairs))
print("Main19-relevant candidate pairs:", len(near_pairs_main19))
print(
    "Candidate A paths matched:",
    near_pairs["relative_path_a"].isin(main19_relative_paths).sum()
)
print(
    "Candidate B paths matched:",
    near_pairs["relative_path_b"].isin(main19_relative_paths).sum()
)

display(
    near_pairs_main19[
        [
            "pair_key",
            "label",
            "phash_hamming_distance",
            "relative_path_a",
            "relative_path_b",
            "decision",
            "reason",
            "reviewer",
            "review_date",
            "exclude_image_path",
        ]
    ].head()
)

In [ ]:
# STEP 2: Create duplicate groups using the normalized Main19-relative paths
# relative_path_a and relative_path_b are the stable image identifiers here.

class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


uf = UnionFind()

pair_paths = near_pairs_main19.dropna(
    subset=["relative_path_a", "relative_path_b"]
).copy()

for _, row in pair_paths.iterrows():
    a = str(row["relative_path_a"]).strip().lower()
    b = str(row["relative_path_b"]).strip().lower()
    uf.union(a, b)

all_group_paths = sorted(
    set(pair_paths["relative_path_a"].astype(str).str.strip().str.lower())
    | set(pair_paths["relative_path_b"].astype(str).str.strip().str.lower())
)

group_records = [
    {
        "relative_path_key": image_path,
        "duplicate_group_root": uf.find(image_path),
    }
    for image_path in all_group_paths
]

dup_groups = pd.DataFrame(group_records)

root_to_group_id = {
    root: f"dupgrp_{i:04d}"
    for i, root in enumerate(
        sorted(dup_groups["duplicate_group_root"].unique()),
        start=1
    )
}

dup_groups["duplicate_group_id"] = dup_groups[
    "duplicate_group_root"
].map(root_to_group_id)

dup_groups = dup_groups[
    ["relative_path_key", "duplicate_group_id"]
].drop_duplicates()

print("Main19 candidate pairs:", len(near_pairs_main19))
print("Images in duplicate groups:", len(dup_groups))
print("Duplicate groups:", dup_groups["duplicate_group_id"].nunique())

display(
    dup_groups
    .sort_values(["duplicate_group_id", "relative_path_key"])
    .head(20)
)

In [ ]:
# STEP 3: Attach duplicate-group IDs to the locked Main19 manifest

locked_main19["relative_path_key"] = to_dataset_relative_path(
    locked_main19["relative_path"]
)

locked_main19_with_groups = locked_main19.merge(
    dup_groups,
    on="relative_path_key",
    how="left",
    validate="one_to_one",
)

print("Main19 images:", len(locked_main19_with_groups))
print(
    "Main19 images belonging to candidate duplicate groups:",
    locked_main19_with_groups["duplicate_group_id"].notna().sum()
)

display(
    locked_main19_with_groups.loc[
        locked_main19_with_groups["duplicate_group_id"].notna(),
        [
            "relative_path",
            "class_name",
            "historical_split",
            "duplicate_group_id",
        ],
    ]
    .sort_values(["duplicate_group_id", "historical_split"])
)

In [ ]:
# STEP 4: Confirm the two reviewed leakage exclusions are absent

excluded_near_dup_df["relative_path_key"] = to_dataset_relative_path(
    excluded_near_dup_df["relative_path"]
)

excluded_paths = set(excluded_near_dup_df["relative_path_key"])
final_main19_paths = set(locked_main19_with_groups["relative_path_key"])

still_present = excluded_paths & final_main19_paths

print("Confirmed leakage exclusions recorded:", len(excluded_paths))
print("Confirmed leakage images still in final Main19:", len(still_present))

if still_present:
    print("\nERROR — excluded paths still present:")
    for path in sorted(still_present):
        print(path)
    raise ValueError("Confirmed near-duplicate leakage is still present.")
else:
    print("\nPASS — all confirmed leakage images are absent from final Main19.")

In [ ]:
print("review_progress columns:", review_progress_df.columns.tolist())
print("review_decisions columns:", review_decisions_df.columns.tolist())
print("excluded_confirmed_near_dup columns:", excluded_near_dup_df.columns.tolist())

In [ ]:
def harmonize_review_decisions(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    image_a_col = first_existing(
        out,
        ["image_a_path", "train_image_path", "image_path_a"],
        required=True,
        df_name="review_decisions",
    )

    image_b_col = first_existing(
        out,
        ["image_b_path", "val_image_path", "image_path_b"],
        required=True,
        df_name="review_decisions",
    )

    decision_col = first_existing(
        out,
        ["review_decision", "decision"],
        required=True,
        df_name="review_decisions",
    )

    reason_col = first_existing(
        out,
        ["review_reason", "reason"],
        required=False,
        df_name="review_decisions",
    )

    reviewer_col = first_existing(
        out,
        ["reviewer"],
        required=False,
        df_name="review_decisions",
    )

    review_date_col = first_existing(
        out,
        ["review_date", "date"],
        required=False,
        df_name="review_decisions",
    )

    result = pd.DataFrame({
        "relative_path_a": to_dataset_relative_path(out[image_a_col]),
        "relative_path_b": to_dataset_relative_path(out[image_b_col]),
        "decision": out[decision_col].astype(str).str.strip().str.upper(),
    })

    if reason_col is not None:
        result["reason"] = out[reason_col]
    else:
        result["reason"] = pd.NA

    if reviewer_col is not None:
        result["reviewer"] = out[reviewer_col]
    else:
        result["reviewer"] = pd.NA

    if review_date_col is not None:
        result["review_date"] = out[review_date_col]
    else:
        result["review_date"] = pd.NA

    result["pair_key"] = result.apply(
        lambda row: make_pair_key(
            row["relative_path_a"],
            row["relative_path_b"],
        ),
        axis=1,
    )

    # Treat these as not actually reviewed.
    pending_values = {"", "NAN", "NONE", "PENDING", "UNREVIEWED"}
    result.loc[
        result["decision"].isin(pending_values),
        "decision"
    ] = pd.NA

    return result


review_decisions_h = harmonize_review_decisions(review_progress_df)

print("Review-decision shape:", review_decisions_h.shape)
print("Missing pair keys:", review_decisions_h["pair_key"].isna().sum())
print("Actual reviewed rows:", review_decisions_h["decision"].notna().sum())

display(review_decisions_h.head())
display(review_decisions_h["decision"].value_counts(dropna=False))

In [ ]:
def make_pair_key(a, b):
    a = str(a)
    b = str(b)
    return "||".join(sorted([a, b]))

for df in [near_pairs_main19, review_decisions_h]:
    if "image_id_a" in df.columns and "image_id_b" in df.columns:
        df["pair_key"] = df.apply(lambda r: make_pair_key(r["image_id_a"], r["image_id_b"]), axis=1)

near_pairs_main19["pair_key"].head()

In [ ]:
review_pairs = (
    review_decisions_h[
        ["pair_key", "decision", "reason", "reviewer", "review_date"]
    ]
    .dropna(subset=["pair_key"])
    .drop_duplicates(subset=["pair_key"], keep="last")
)

near_pairs_review = near_pairs_main19.merge(
    review_pairs,
    on="pair_key",
    how="left",
    validate="one_to_one",
)

print("Main19 candidate pairs:", len(near_pairs_review))
print("Pairs with reviewed decisions:", near_pairs_review["decision"].notna().sum())

display(near_pairs_review[[
    "label",
    "phash_hamming_distance",
    "relative_path_a",
    "relative_path_b",
    "decision",
    "reason",
    "reviewer",
]].head())

display(near_pairs_review["decision"].value_counts(dropna=False))

In [ ]:
confirmed_keywords = {
    "exact/transformed copy",
    "same leaf",
    "same capture",
    "confirmed duplicate",
    "confirmed leakage",
    "exclude",
    "near duplicate confirmed",
}

def is_confirmed_leakage(decision_text):
    if pd.isna(decision_text):
        return False
    txt = str(decision_text).strip().lower()
    return any(k in txt for k in confirmed_keywords)

near_pairs_review["confirmed_leakage"] = near_pairs_review["decision"].apply(is_confirmed_leakage)

print("Confirmed leakage pairs:", near_pairs_review["confirmed_leakage"].sum())
display(near_pairs_review.loc[near_pairs_review["confirmed_leakage"]].head(20))

In [ ]:
unresolved_pairs = near_pairs_review.loc[near_pairs_review["decision"].isna()].copy()

unresolved_export_cols = [
    c for c in [
        "image_id_a", "image_id_b", "relative_path_a", "relative_path_b",
        "phash_hamming_distance", "split_a", "split_b", "pair_key"
    ] if c in unresolved_pairs.columns
]

unresolved_export = unresolved_pairs[unresolved_export_cols].copy()
unresolved_export["manual_decision"] = ""
unresolved_export["manual_reason"] = ""
unresolved_export["reviewer"] = ""
unresolved_export["review_date"] = ""

unresolved_path = WORK_ROOT / "unresolved_near_duplicate_pairs_for_manual_review.csv"
unresolved_export.to_csv(unresolved_path, index=False)

print("Unresolved pairs exported to:", unresolved_path)
print("Count:", len(unresolved_export))
display(unresolved_export.head())

In [ ]:
def path_from_rel(rel_path):
    if pd.isna(rel_path):
        return None
    p = resolve_image_path(rel_path)
    return p

def show_pair(row):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, rel_col, title in zip(
        axes,
        ["relative_path_a", "relative_path_b"],
        ["A", "B"]
    ):
        rel = row.get(rel_col, None)
        p = path_from_rel(rel)
        ax.axis("off")
        if p is None or not Path(p).exists():
            ax.set_title(f"{title}: path not found")
            continue
        img = Image.open(p).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{title}\n{rel}")
    plt.tight_layout()
    plt.show()

if len(unresolved_pairs) > 0:
    sample_row = unresolved_pairs.iloc[0].to_dict()
    show_pair(sample_row)

In [ ]:
REVIEW_PANEL_DIR = WORK_ROOT / "review_panels"
REVIEW_PANEL_DIR.mkdir(parents=True, exist_ok=True)

def save_pair_panel(row, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    titles = []
    for ax, rel_col, side in zip(axes, ["relative_path_a", "relative_path_b"], ["A", "B"]):
        rel = row.get(rel_col, None)
        p = path_from_rel(rel)
        ax.axis("off")
        if p is None or not Path(p).exists():
            ax.set_title(f"{side}: missing")
            continue
        img = Image.open(p).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{side}\n{Path(rel).name}")
    fig.suptitle(f"Pair key: {row.get('pair_key','')}\nHamming: {row.get('phash_hamming_distance', np.nan)}")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

preview_n = min(20, len(unresolved_pairs))
for i, (_, row) in enumerate(unresolved_pairs.head(preview_n).iterrows(), start=1):
    save_pair_panel(row.to_dict(), REVIEW_PANEL_DIR / f"pair_{i:03d}.png")

print("Saved preview panels:", preview_n)
print("Directory:", REVIEW_PANEL_DIR)

In [ ]:
UNRESOLVED_COUNT = len(unresolved_pairs)
print("UNRESOLVED REVIEW COUNT:", UNRESOLVED_COUNT)

ALLOW_RETRAIN_WITH_UNRESOLVED = False

if UNRESOLVED_COUNT > 0 and not ALLOW_RETRAIN_WITH_UNRESOLVED:
    raise RuntimeError(
        "Manual duplicate review is incomplete. Resolve unresolved pairs before retraining, "
        "or explicitly set ALLOW_RETRAIN_WITH_UNRESOLVED=True for dry-run debugging only."
    )

In [ ]:
confirmed_pairs = near_pairs_review.loc[near_pairs_review["confirmed_leakage"]].copy()

images_to_exclude = set()

if not excluded_near_dup_df.empty:
    possible_cols = [c for c in ["image_id", "excluded_image_id", "id"] if c in excluded_near_dup_df.columns]
    if possible_cols:
        images_to_exclude |= set(excluded_near_dup_df[possible_cols[0]].astype(str).tolist())

for _, row in confirmed_pairs.iterrows():
    a = str(row["image_id_a"])
    b = str(row["image_id_b"])

    # Prefer excluding calibration/internal-test-side item only if split info exists;
    # else exclude B deterministically to preserve reproducibility.
    split_a = str(row["split_a"]).lower() if "split_a" in row and pd.notna(row["split_a"]) else ""
    split_b = str(row["split_b"]).lower() if "split_b" in row and pd.notna(row["split_b"]) else ""

    if split_b in {"calibration", "internal_test", "test", "val", "validation"}:
        images_to_exclude.add(b)
    elif split_a in {"calibration", "internal_test", "test", "val", "validation"}:
        images_to_exclude.add(a)
    else:
        images_to_exclude.add(b)

print("Images to exclude from confirmed near-duplicate leakage:", len(images_to_exclude))
list(sorted(images_to_exclude))[:10]

In [ ]:
audited_main19 = locked_main19.copy()
audited_main19["excluded_due_to_confirmed_near_dup"] = audited_main19["image_id"].astype(str).isin(images_to_exclude)
audited_main19["excluded_reason"] = np.where(
    audited_main19["excluded_due_to_confirmed_near_dup"],
    "confirmed_near_duplicate_leakage",
    ""
)

audited_main19_kept = audited_main19.loc[~audited_main19["excluded_due_to_confirmed_near_dup"]].copy()

print("Original locked Main19:", len(locked_main19))
print("Excluded from confirmed near-duplicate leakage:", audited_main19["excluded_due_to_confirmed_near_dup"].sum())
print("Audited kept Main19:", len(audited_main19_kept))

display(audited_main19_kept["historical_split"].value_counts())

In [ ]:
audited_main19_kept = audited_main19_kept.merge(
    dup_groups,
    on="image_id",
    how="left"
)

audited_main19_kept["duplicate_group_id"] = audited_main19_kept["duplicate_group_id"].fillna(
    audited_main19_kept["image_id"].apply(lambda x: f"solo_{x}")
)

print("Rows:", len(audited_main19_kept))
print("Unique duplicate_group_id:", audited_main19_kept["duplicate_group_id"].nunique())
display(audited_main19_kept.head())

In [ ]:
revised_train_pool = audited_main19_kept.loc[audited_main19_kept["historical_split"] == "train"].copy()
revised_temp_source = audited_main19_kept.loc[audited_main19_kept["historical_split"] == "calibration"].copy()
revised_internal_test = audited_main19_kept.loc[audited_main19_kept["historical_split"] == "internal_test"].copy()

print("Revised train pool:", revised_train_pool.shape)
print("Revised original calibration pool:", revised_temp_source.shape)
print("Revised internal test:", revised_internal_test.shape)

In [ ]:
group_df = (
    revised_train_pool.groupby("duplicate_group_id")
    .agg(
        class_name=("class_name", lambda x: x.mode().iat[0]),
        n_images=("image_id", "count")
    )
    .reset_index()
)

print(group_df.shape)
display(group_df.head())
display(group_df["class_name"].value_counts().head())

In [ ]:
train_groups = []
esval_groups = []

for class_name, sub in group_df.groupby("class_name"):
    groups = sub["duplicate_group_id"].tolist()

    if len(groups) == 1:
        train_groups.extend(groups)
        continue

    n_val = max(1, round(0.10 * len(groups)))
    rng = np.random.default_rng(SEED + abs(hash(class_name)) % 10000)
    shuffled = groups.copy()
    rng.shuffle(shuffled)

    es_part = shuffled[:n_val]
    tr_part = shuffled[n_val:]

    if len(tr_part) == 0:
        tr_part = es_part[:1]
        es_part = es_part[1:]

    train_groups.extend(tr_part)
    esval_groups.extend(es_part)

train_groups = set(train_groups)
esval_groups = set(esval_groups)

print("Train groups:", len(train_groups))
print("ES-Val groups:", len(esval_groups))
print("Overlap:", len(train_groups & esval_groups))

In [ ]:
revised_train_final = revised_train_pool.loc[revised_train_pool["duplicate_group_id"].isin(train_groups)].copy()
es_val_final = revised_train_pool.loc[revised_train_pool["duplicate_group_id"].isin(esval_groups)].copy()

print("Revised train final:", revised_train_final.shape)
print("ES-Val final:", es_val_final.shape)
print("Group overlap:", len(set(revised_train_final["duplicate_group_id"]) & set(es_val_final["duplicate_group_id"])))

In [ ]:
def overlap_count(df1, df2, col="image_id"):
    return len(set(df1[col].astype(str)) & set(df2[col].astype(str)))

print("Train vs ES-Val image overlap:", overlap_count(revised_train_final, es_val_final))
print("Train vs Calibration overlap:", overlap_count(revised_train_final, revised_temp_source))
print("Train vs Internal-Test overlap:", overlap_count(revised_train_final, revised_internal_test))
print("ES-Val vs Calibration overlap:", overlap_count(es_val_final, revised_temp_source))
print("ES-Val vs Internal-Test overlap:", overlap_count(es_val_final, revised_internal_test))
print("Calibration vs Internal-Test overlap:", overlap_count(revised_temp_source, revised_internal_test))

def dup_group_overlap(df1, df2):
    return len(set(df1["duplicate_group_id"]) & set(df2["duplicate_group_id"]))

print("Train vs ES-Val group overlap:", dup_group_overlap(revised_train_final, es_val_final))
print("Train vs Calibration group overlap:", dup_group_overlap(revised_train_final, revised_temp_source))
print("Train vs Internal-Test group overlap:", dup_group_overlap(revised_train_final, revised_internal_test))
print("ES-Val vs Calibration group overlap:", dup_group_overlap(es_val_final, revised_temp_source))
print("ES-Val vs Internal-Test group overlap:", dup_group_overlap(es_val_final, revised_internal_test))
print("Calibration vs Internal-Test group overlap:", dup_group_overlap(revised_temp_source, revised_internal_test))

In [ ]:
def class_count_table(df, split_name):
    out = df["class_name"].value_counts().sort_index().rename(split_name).to_frame()
    out["label_index"] = out.index.map(class_to_index_main19)
    return out.reset_index().rename(columns={"index": "class_name"})

count_train = class_count_table(revised_train_final, "train_n")
count_es = class_count_table(es_val_final, "es_val_n")
count_cal = class_count_table(revised_temp_source, "calibration_n")
count_test = class_count_table(revised_internal_test, "internal_test_n")

count_summary = count_train.merge(count_es, on=["class_name", "label_index"], how="outer")
count_summary = count_summary.merge(count_cal, on=["class_name", "label_index"], how="outer")
count_summary = count_summary.merge(count_test, on=["class_name", "label_index"], how="outer")
count_summary = count_summary.fillna(0).sort_values("label_index")

display(count_summary)

In [ ]:
def finalize_manifest(df, revised_split_name):
    out = df.copy()
    out["revised_split"] = revised_split_name
    keep_cols = [
        "image_id", "relative_path", "resolved_path", "class_name", "label_index",
        "historical_split", "revised_split", "duplicate_group_id"
    ]
    return out[keep_cols].sort_values(["label_index", "image_id"]).reset_index(drop=True)

nb12_train = finalize_manifest(revised_train_final, "train")
nb12_esval = finalize_manifest(es_val_final, "es_val")
nb12_calibration = finalize_manifest(revised_temp_source, "calibration")
nb12_internal_test = finalize_manifest(revised_internal_test, "internal_test")

nb12_train.to_csv(WORK_ROOT / "nb12_main19_train.csv", index=False)
nb12_esval.to_csv(WORK_ROOT / "nb12_main19_es_val.csv", index=False)
nb12_calibration.to_csv(WORK_ROOT / "nb12_main19_calibration_source.csv", index=False)
nb12_internal_test.to_csv(WORK_ROOT / "nb12_main19_internal_test.csv", index=False)
count_summary.to_csv(WORK_ROOT / "nb12_split_counts_by_class.csv", index=False)

print("Saved manifests to:", WORK_ROOT)

In [ ]:
qc_report = {
    "seed": SEED,
    "device_at_manifest_stage": DEVICE,
    "original_locked_main19_n": int(len(locked_main19)),
    "excluded_confirmed_near_duplicate_n": int(audited_main19["excluded_due_to_confirmed_near_dup"].sum()),
    "audited_main19_kept_n": int(len(audited_main19_kept)),
    "nb12_train_n": int(len(nb12_train)),
    "nb12_es_val_n": int(len(nb12_esval)),
    "nb12_calibration_source_n": int(len(nb12_calibration)),
    "nb12_internal_test_n": int(len(nb12_internal_test)),
    "unresolved_duplicate_review_n": int(UNRESOLVED_COUNT),
    "train_esval_image_overlap": int(overlap_count(revised_train_final, es_val_final)),
    "train_esval_group_overlap": int(dup_group_overlap(revised_train_final, es_val_final)),
}

with open(WORK_ROOT / "nb12_manifest_qc_report.json", "w", encoding="utf-8") as f:
    json.dump(qc_report, f, indent=2)

qc_report

In [ ]:
class Main19Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

        missing = self.df["resolved_path"].isna().sum()
        if missing > 0:
            raise ValueError(f"Dataset has {missing} unresolved paths.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["resolved_path"]).convert("RGB")
        label = int(row["label_index"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["image_id"]

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = Main19Dataset(nb12_train, transform=train_transform)
esval_ds = Main19Dataset(nb12_esval, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
esval_loader = DataLoader(esval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_ds), len(esval_ds)

In [ ]:
num_classes = len(class_to_index_main19)
train_counts = nb12_train["label_index"].value_counts().sort_index()

weights = []
for i in range(num_classes):
    count = int(train_counts.get(i, 0))
    if count == 0:
        raise ValueError(f"Class index {i} has zero samples in revised train.")
    weights.append(len(nb12_train) / (num_classes * count))

class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
class_weights

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights_enum = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights_enum)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[6, 13], gamma=0.5)

print(model.classifier)

In [ ]:
from sklearn.metrics import f1_score

def run_epoch(model, loader, criterion=None, optimizer=None, train=False):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels) if criterion is not None else None

            if train:
                loss.backward()
                optimizer.step()

        if loss is not None:
            total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = (np.array(y_true) == np.array(y_pred)).mean()

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "macro_f1": macro_f1,
    }

In [ ]:
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4

history = []
best_state = None
best_epoch = None
best_macro_f1 = -np.inf
epochs_without_improve = 0

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion=criterion, optimizer=optimizer, train=True)
    esval_metrics = run_epoch(model, esval_loader, criterion=criterion, optimizer=None, train=False)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "esval_loss": esval_metrics["loss"],
        "esval_accuracy": esval_metrics["accuracy"],
        "esval_macro_f1": esval_metrics["macro_f1"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    improved = esval_metrics["macro_f1"] > best_macro_f1
    if improved:
        best_macro_f1 = esval_metrics["macro_f1"]
        best_epoch = epoch
        best_state = {
            "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "epoch": epoch,
            "esval_macro_f1": best_macro_f1,
        }
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} "
        f"train_f1={row['train_macro_f1']:.4f} | "
        f"esval_loss={row['esval_loss']:.4f} "
        f"esval_f1={row['esval_macro_f1']:.4f}"
    )

    if epochs_without_improve >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

elapsed = time.time() - start_time
print("Training time (sec):", round(elapsed, 2))
print("Best epoch:", best_epoch)
print("Best ES-Val macro-F1:", best_macro_f1)

In [ ]:
history_df = pd.DataFrame(history)
history_path = WORK_ROOT / "nb12_training_history.csv"
history_df.to_csv(history_path, index=False)

checkpoint_path = WORK_ROOT / "best_efficientnet_b0_main19_nb12.pt"
torch.save(best_state, checkpoint_path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checkpoint_hash = sha256_file(checkpoint_path)

print("History saved:", history_path)
print("Checkpoint saved:", checkpoint_path)
print("Checkpoint SHA256:", checkpoint_hash)

In [ ]:
training_config = {
    "seed": SEED,
    "device": DEVICE,
    "model_name": "efficientnet_b0",
    "pretrained_weights": "IMAGENET1K_V1",
    "num_classes": num_classes,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": {
        "type": "MultiStepLR",
        "milestones": [6, 13],
        "gamma": 0.5
    },
    "loss": "CrossEntropyLoss(class_weighted)",
    "class_weights_source": "nb12 revised train only",
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "selection_metric": "esval_macro_f1",
    "best_epoch": int(best_epoch),
    "best_esval_macro_f1": float(best_macro_f1),
    "train_manifest": str(WORK_ROOT / "nb12_main19_train.csv"),
    "esval_manifest": str(WORK_ROOT / "nb12_main19_es_val.csv"),
    "calibration_source_manifest": str(WORK_ROOT / "nb12_main19_calibration_source.csv"),
    "internal_test_manifest": str(WORK_ROOT / "nb12_main19_internal_test.csv"),
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": checkpoint_hash,
    "historical_internal_test_used_for_selection": False,
    "plantdoc_used": False,
}

with open(WORK_ROOT / "nb12_training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2)

training_config

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["esval_loss"], label="esval_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="train_macro_f1")
plt.plot(history_df["epoch"], history_df["esval_macro_f1"], label="esval_macro_f1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.legend()

plt.tight_layout()
plt.savefig(WORK_ROOT / "nb12_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
summary_lines = [
    "# Notebook 12 Summary",
    "",
    "## Duplicate review and audited partitioning",
    f"- Original locked Main19 images: {len(locked_main19)}",
    f"- Confirmed near-duplicate exclusions applied: {int(audited_main19['excluded_due_to_confirmed_near_dup'].sum())}",
    f"- Audited kept Main19 images: {len(audited_main19_kept)}",
    f"- Unresolved duplicate-review pairs at run time: {UNRESOLVED_COUNT}",
    "",
    "## Revised split sizes",
    f"- Train: {len(nb12_train)}",
    f"- ES-Val: {len(nb12_esval)}",
    f"- Calibration source (for Notebook 13 only): {len(nb12_calibration)}",
    f"- Internal test (held, not used here): {len(nb12_internal_test)}",
    "",
    "## Retraining",
    f"- Device: {DEVICE}",
    f"- Best epoch: {best_epoch}",
    f"- Best ES-Val macro-F1: {best_macro_f1:.6f}",
    f"- Checkpoint: {checkpoint_path.name}",
    f"- Checkpoint SHA256: {checkpoint_hash}",
    "",
    "## Restrictions honored",
    "- No temperature fitting performed.",
    "- No conformal calibration performed.",
    "- No internal-test evaluation performed.",
    "- No PlantDoc evaluation performed.",
]

summary_path = WORK_ROOT / "nb12_summary_report.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print(summary_path.read_text(encoding="utf-8"))

In [ ]:
for p in sorted(WORK_ROOT.rglob("*")):
    print(p)

In [ ]:
print(
    "Notebook 12 completed: duplicate-review integration, audited partition export, "
    "ES-Val creation, and EfficientNet-B0 retraining are done.\n"
    "Notebook 13 should begin from the exported Notebook-12 manifests and checkpoint.\n"
    "Do not run internal-test or PlantDoc evaluation in this notebook."
)